# Ocean Response to Cyclone Mocha — Bay of Bengal, May 2023

Tropical Cyclone Mocha made landfall near **Sittwe, Myanmar** on **14 May 2023** as one of
the strongest storms ever recorded in the Bay of Bengal (Category 5 equivalent).

This notebook analyses how key physical and biogeochemical ocean variables changed across
the **Bay of Bengal and Andaman Sea** during May 2023 by following the MAGICA course workflow:

1. **Lazy metadata-first access** — open Copernicus Marine datasets without downloading data
2. **Inspect metadata** — confirm shape, coordinates and chunk layout before any compute
3. **Subset** spatially (AOI) and temporally (May 2023)
4. **Write to Icechunk** — versioned zarr store in personal S3 bucket
5. **Read back lazily** — re-open from Icechunk without loading data
6. **Analyse** — load area-averaged timeseries into memory for plotting
7. **Visualise** — spatial maps before, during, and after landfall

## Variables monitored

| Variable | CMEMS product |
|---|---|
| Sea surface temperature (`thetao`) | `cmems_mod_glo_phy_my_0.083deg_P1D-m` |
| Salinity (`so`) | `cmems_mod_glo_phy_my_0.083deg_P1D-m` |
| Significant wave height (`VHM0`) | `cmems_mod_glo_wav_my_0.2deg_PT3H-i` |
| Chlorophyll (`chl`) | `cmems_mod_glo_bgc_my_0.25deg_P1D-m` |
| Net primary production (`nppv`) | `cmems_mod_glo_bgc_my_0.25deg_P1D-m` |
| Nitrate (`no3`) | `cmems_mod_glo_bgc_my_0.25deg_P1D-m` |
| Phosphate (`po4`) | `cmems_mod_glo_bgc_my_0.25deg_P1D-m` |

All products are **reanalysis** (measurements-based), covering 1993–present.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import xarray as xr
import icechunk
import copernicusmarine
import hvplot.xarray
import panel as pn
from dotenv import load_dotenv

pn.extension()
print("All imports OK")

In [ ]:
# Credentials
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)
storage_endpoint = os.environ['ENDPOINT_URL']
BUCKET = 'raquelcapella'

# Area of Interest: Bay of Bengal + Andaman Sea (Cyclone Mocha track)
#bbox = [85.0, 10.0, 100.0, 25.0]  # [lon_min, lat_min, lon_max, lat_max]
bbox= [90.9, 17.5, 94.6, 21.3]

# Full month of May 2023 — captures pre-storm baseline, cyclone peak, and recovery
TIME_START = "2023-05-01"
TIME_END   = "2023-05-31"
LANDFALL   = "2023-05-14"

print(f"AOI: lon [{bbox[0]}, {bbox[2]}]  lat [{bbox[1]}, {bbox[3]}]")
print(f"Period: {TIME_START} → {TIME_END}")
print(f"Cyclone Mocha landfall: {LANDFALL}")

## 1. Metadata-first: open Copernicus Marine datasets lazily

`copernicusmarine.open_dataset` returns a **dask-backed xarray Dataset** — the data catalogue
metadata (shape, coordinates, chunk layout) is available immediately without transferring
any actual data values.  We inspect this metadata first to understand what we are working
with before triggering any compute.

We use three reanalysis products:
- **Physics** `cmems_mod_glo_phy_my_0.083deg_P1D-m` — daily 0.083°, SST + salinity (1993–present)
- **Waves** `cmems_mod_glo_wav_my_0.2deg_PT3H-i` — 3-hourly 0.2°, significant wave height (1993–present)
- **BGC reanalysis** `cmems_mod_glo_bgc_my_0.25deg_P1D-m` — daily 0.25°, chl/no3/po4/nppv (1993–present)

In [ ]:
%%time
# Physics reanalysis — daily, 0.083° global grid
# open_dataset is lazy: only catalogue metadata is fetched here
ds_phy_full = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "so"],
)
ds_phy_full

In [ ]:
%%time
# Wave reanalysis — 3-hourly, 0.2° global grid, covers 1993-present
# VHM0 = spectral significant wave height (combines wind sea + swell)
ds_wav_full = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_wav_my_0.2deg_PT3H-i",
    variables=["VHM0"],
)
ds_wav_full

In [ ]:
%%time
# BGC multi-year reanalysis — daily, 0.25° global grid, covers 1993-present
# chl, no3, po4, nppv all available for May 2023
ds_bgc_full = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_bgc_my_0.25deg_P1D-m",
    variables=["chl", "no3", "po4", "nppv"],
)

print("BGC reanalysis:")
display(ds_bgc_full)

In [ ]:
# ── Metadata inspection — zero data downloaded ────────────────────────────────
# Shape, dtype, chunks and coordinate ranges come from xarray/dask metadata only.

for label, ds in [
    ("Physics (phy)", ds_phy_full),
    ("Waves (wav)", ds_wav_full),
    ("BGC reanalysis (bgc)", ds_bgc_full),
]:
    print(f"── {label} ──")
    print(f"  variables : {list(ds.data_vars)}")
    print(f"  time      : {ds.time.values[0]} → {ds.time.values[-1]}  ({len(ds.time)} steps)")
    print(f"  latitude  : {float(ds.latitude.min()):.2f} → {float(ds.latitude.max()):.2f}")
    print(f"  longitude : {float(ds.longitude.min()):.2f} → {float(ds.longitude.max()):.2f}")
    for var in ds.data_vars:
        da = ds[var]
        total_gb = da.nbytes / 1e9
        print(f"  {var:10s}: shape {da.shape}  dtype {da.dtype}  ~{total_gb:.0f} GB uncompressed")
    print()

print("^ All from dask/xarray metadata — zero data values fetched.")

### 1b. Subset to AOI and May 2023

`.sel()` on a dask-backed dataset is still **lazy** — it narrows the coordinate range
without loading any data.  The resulting datasets are much smaller and fast to write.

In [ ]:
sel = dict(
    latitude=slice(bbox[1], bbox[3]),
    longitude=slice(bbox[0], bbox[2]),
    time=slice(TIME_START, TIME_END),
)

# Surface layer only — depth index 0 (~0.5 m). Still lazy.
ds_phy = ds_phy_full.sel(**sel).isel(depth=0, drop=True)

# Wave product has no depth dimension — subset directly, then resample 3-hourly → daily mean
ds_wav = ds_wav_full.sel(**sel).resample(time="1D").mean()

# BGC reanalysis also has multiple depth levels — take surface (index 0, ~0.5 m)
ds_bgc = ds_bgc_full.sel(**sel).isel(depth=0, drop=True)

for label, ds in [("Physics subset", ds_phy), ("Waves subset", ds_wav), ("BGC subset", ds_bgc)]:
    print(f"── {label} ──")
    print(f"  time      : {ds.time.values[0]} → {ds.time.values[-1]}  ({len(ds.time)} steps)")
    for var in ds.data_vars:
        mb = ds[var].nbytes / 1e6
        print(f"  {var:10s}: shape {ds[var].shape}  ~{mb:.1f} MB")
    print()

## 2. Write to Icechunk

We write each subset into a personal Icechunk repository on the S3-compatible object store.
The try/except pattern handles both first-run (create) and re-run (open-existing) cases cleanly.

In [ ]:
def get_or_create_repo(prefix):
    """Open existing Icechunk repo or create a fresh one."""
    storage = icechunk.s3_storage(
        bucket=BUCKET,
        prefix=prefix,
        from_env=True,
        endpoint_url=storage_endpoint,
        region='not-used',
        force_path_style=True,
    )
    try:
        repo = icechunk.Repository.open(storage)
        print(f"  Opened existing repo: s3://{BUCKET}/{prefix}")
    except Exception:
        repo = icechunk.Repository.create(storage)
        print(f"  Created new repo:     s3://{BUCKET}/{prefix}")
    return repo

In [ ]:
%%time
print("Writing physics store (SST + salinity)...")
repo_phy = get_or_create_repo("icechunk/myanmar_mocha_physics")
session_phy = repo_phy.writable_session("main")
ds_phy.drop_encoding().to_zarr(session_phy.store, mode="w")
snap_phy = session_phy.commit("Physics (thetao, so) — Cyclone Mocha AOI, May 2023")
print(f"  Committed [{snap_phy[:8]}…]: s3://{BUCKET}/icechunk/myanmar_mocha_physics")

In [ ]:
%%time
print("Writing wave store (VHM0)...")
repo_wav = get_or_create_repo("icechunk/myanmar_mocha_waves")
session_wav = repo_wav.writable_session("main")
# resample() produces irregular dask chunks at AOI boundaries — rechunk to uniform before writing
ds_wav.chunk({"time": -1, "latitude": -1, "longitude": -1}).drop_encoding().to_zarr(session_wav.store, mode="w")
snap_wav = session_wav.commit("Waves (VHM0 daily mean) — Cyclone Mocha AOI, May 2023")
print(f"  Committed [{snap_wav[:8]}…]: s3://{BUCKET}/icechunk/myanmar_mocha_waves")

In [ ]:
%%time
print("Writing BGC store (chl, nppv, no3, po4)...")
repo_bgc = get_or_create_repo("icechunk/myanmar_mocha_bgc")
session_bgc = repo_bgc.writable_session("main")
ds_bgc.drop_encoding().to_zarr(session_bgc.store, mode="w")
snap_bgc = session_bgc.commit("BGC (chl, nppv, no3, po4) — Cyclone Mocha AOI, May 2023")
print(f"  Committed [{snap_bgc[:8]}…]: s3://{BUCKET}/icechunk/myanmar_mocha_bgc")

## 3. Read back from Icechunk — still lazy

Re-opening from Icechunk returns dask-backed arrays.  The coordinate values and array
shapes are available from zarr metadata without loading any data values — this is the
same lazy pattern as opening from Copernicus Marine.

In [ ]:
# Re-open from Icechunk — consolidated=False is required for zarr v3 stores
ic_phy = xr.open_zarr(repo_phy.readonly_session("main").store, consolidated=False)
ic_wav = xr.open_zarr(repo_wav.readonly_session("main").store, consolidated=False)
ic_bgc = xr.open_zarr(repo_bgc.readonly_session("main").store, consolidated=False)

print("Physics store (from Icechunk):")
display(ic_phy)
print("\nWave store (from Icechunk):")
display(ic_wav)
print("\nBGC store (from Icechunk):")
display(ic_bgc)

print("\n^ Arrays are dask-backed — no data values loaded yet.")

## 4. Timeseries analysis — load into memory

We compute the **area-averaged** value of each variable for every day in May 2023.
This is where `.load()` is called — the actual data is fetched from the Icechunk store
and brought into RAM.  The arrays are small (~31 time steps × scalar = negligible).

The vertical dashed line marks cyclone landfall on **14 May 2023**.

In [ ]:
%%time
# Spatial mean → timeseries.  .load() triggers the actual data fetch.
spatial_dims = ["latitude", "longitude"]

ts = {}
for var in ["thetao", "so"]:
    ts[var] = ic_phy[var].mean(dim=spatial_dims).load()
    print(f"  {var}: loaded {ts[var].shape[0]} time steps")

ts["VHM0"] = ic_wav["VHM0"].mean(dim=spatial_dims).load()
print(f"  VHM0: loaded {ts['VHM0'].shape[0]} time steps")

for var in ["chl", "nppv", "no3", "po4"]:
    ts[var] = ic_bgc[var].mean(dim=spatial_dims).load()
    print(f"  {var}: loaded {ts[var].shape[0]} time steps")

In [ ]:
import pandas as pd
import holoviews as hv

landfall_line = hv.VLine(pd.Timestamp(LANDFALL)).opts(
    color="red", line_dash="dashed", line_width=1.5
)

p_wav = (
    ts["VHM0"].hvplot(label="Sig. wave height (m)", color="#ff7f0e", ylabel="m")
    * landfall_line
).opts(title="Significant Wave Height (VHM0) — Bay of Bengal, May 2023", width=700, height=280)

p_wav

In [ ]:
p_sst = (
    ts["thetao"].hvplot(label="SST (°C)", color="#d62728", ylabel="°C")
    * landfall_line
).opts(title="Sea Surface Temperature — Bay of Bengal, May 2023", width=700, height=280)

p_sal = (
    ts["so"].hvplot(label="Salinity (PSU)", color="#1f77b4", ylabel="PSU")
    * landfall_line
).opts(title="Sea Surface Salinity", width=700, height=280)

(p_sst + p_sal).cols(1)

In [ ]:
plot_cfg = [
    ("no3",  "Nitrate (mmol m⁻³)",             "#9467bd"),
    ("po4",  "Phosphate (mmol m⁻³)",           "#e377c2"),
    ("chl",  "Chlorophyll (mg m⁻³)",          "#2ca02c"),
    ("nppv", "Net Primary Production (mg C m⁻³ d⁻¹)", "#8c564b")
]

plots = [
    (
        ts[var].hvplot(label=ylabel, color=col, ylabel=ylabel.split(" ")[0])
        * landfall_line
    ).opts(title=ylabel, width=700, height=230)
    for var, ylabel, col in plot_cfg
]

hv.Layout(plots).cols(1)

## 5. Interactive spatial maps — full May 2023 with time slider

Each map covers all 31 days in May 2023. Use the **time slider** beneath each map to step through daily snapshots and track the cyclone's spatial imprint across the Bay of Bengal.

Key reference dates:
- **2023-05-10** — pre-cyclone baseline
- **2023-05-14** — landfall near Sittwe (Category 5)
- **2023-05-20** — post-cyclone, recovery and bloom

In [ ]:
%%time
wav_all = ic_wav["VHM0"].load()

wav_all_fmt = wav_all.assign_coords(
    time=wav_all.time.dt.strftime("%Y-%m-%d %H:%M")                                                                                               
) 

wav_map = wav_all_fmt.hvplot(
    x="longitude", y="latitude",
    groupby="time", 
    clim=(0, 6), 
    cmap="Blues",
    geo=True, tiles="OSM", rasterize=True,
    frame_width=500, colorbar=True,
    widget_type="scrubber", widget_location="bottom",
    title="Significant Wave Height (m) — Bay of Bengal, May 2023",
)

wav_map

In [ ]:
%%time
# Load the full time series into memory once, then hvplot renders each time step on demand
sst_all = ic_phy["thetao"].load()

sst_map = sst_all.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    cmap="RdBu_r", clim=(29, 33),
    geo=True, tiles="OSM", rasterize=True,
    frame_width=500, colorbar=True,
    widget_type="scrubber", widget_location="bottom",
    title="Sea Surface Temperature (°C) — Bay of Bengal, May 2023",
)

sst_map

In [ ]:
%%time
sal_all = ic_phy["so"].load()

sal_map = sal_all.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    cmap="inferno_r", clim=(26, 34),
    geo=True, tiles="OSM", rasterize=True,
    frame_width=500, colorbar=True,
    widget_type="scrubber", widget_location="bottom",
    title="Sea Surface Salinity (PSU) — Bay of Bengal, May 2023",
)

sal_map

In [ ]:
%%time
nppv_all = ic_bgc["nppv"].load()
no3_all  = ic_bgc["no3"].load()
po4_all  = ic_bgc["po4"].load()

no3_map = no3_all.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    cmap="PuRd", clim=(2,20),
    geo=True, tiles="OSM", rasterize=True,
    frame_width=500, colorbar=True,
    widget_type="scrubber", widget_location="bottom",
    title="Nitrate (mmol m⁻³) — Bay of Bengal, May 2023",
)

po4_map = po4_all.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    cmap="RdPu",
    geo=True, tiles="OSM", rasterize=True,
    frame_width=500, colorbar=True,
    widget_type="scrubber", widget_location="bottom",
    title="Phosphate (mmol m⁻³) — Bay of Bengal, May 2023",
)

nppv_map = nppv_all.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    cmap="YlOrBr",
    geo=True, tiles="OSM", rasterize=True,
    frame_width=500, colorbar=True,
    widget_type="scrubber", widget_location="bottom",
    title="Net Primary Production (mg C m⁻³ d⁻¹) — Bay of Bengal, May 2023",
)

pn.Column(no3_map, po4_map, nppv_map)

In [ ]:
%%time
chl_all = ic_bgc["chl"].load()

chl_map = chl_all.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    cmap="YlGn",
    geo=True, tiles="OSM", rasterize=True,
    frame_width=500, colorbar=True,
    widget_type="scrubber", widget_location="bottom",
    title="Chlorophyll (mg m⁻³) — Bay of Bengal, May 2023",
)

chl_map

## 6. Icechunk version history

Every `.commit()` creates an immutable, addressable snapshot.  `repo.ancestry()` walks
the commit graph, allowing exact reproducibility and roll-back.

In [ ]:
for label, repo in [("Physics", repo_phy), ("Waves", repo_wav), ("BGC", repo_bgc)]:
    print(f"── {label} store commit history ──")
    for snap in repo.ancestry(branch="main"):
        print(f"  [{snap.written_at}]  {snap.id[:8]}…  {snap.message}")
    print()

## Summary

| Step | Tool | What happened |
|---|---|---|
| Metadata inspection | `copernicusmarine.open_dataset` | Opened three reanalysis datasets lazily — shape and coordinates available instantly, no data downloaded |
| Subset | `xarray.sel` + `isel` + `resample` | Narrowed to Bay of Bengal AOI, May 2023, surface depth; wave data resampled 3-hourly → daily |
| Versioned storage | `icechunk` + `zarr` | Wrote physics, wave, and BGC subsets to personal S3 bucket with commit history |
| Lazy re-read | `xr.open_zarr` | Re-opened from Icechunk without loading data |
| Timeseries analysis | `.mean().load()` | Loaded area-averaged timeseries into memory only when needed |
| Spatial maps | `.sel().load()` | Loaded individual time snapshots for comparison |

### Physical interpretation

Tropical cyclones mix the upper ocean through wind-driven upwelling, bringing cold, nutrient-rich water to the surface. Expected signals in the Bay of Bengal during Cyclone Mocha:

- **SST cooling** to the right of the track (strong wind stress curl)
- **Salinity freshening** near the coast due to heavy rainfall and river runoff
- **Wave height surge** peaking at/just before landfall, decaying rapidly afterward
- **Chlorophyll bloom** 3–10 days post-landfall (lag for photosynthesis response to upwelled nutrients)
- **Net primary production increase** following the bloom, driven by upwelled nitrate and phosphate
- **Nitrate and phosphate increase** at the surface driven by vertical mixing and upwelling